In [ ]:
import numpy as np


def read_sum(file_path):
    pssm = []
    with open(file_path, 'r') as f:
        lines = f.readlines()
        for line in lines:
            if line.startswith(' '):
                pass
            else:
                pssm_row = line.split()[1:]
                pssm.append([float(score) for score in pssm_row])
    return np.array(pssm)


def encode(peptides, encoding_scheme, alphabet):
    
    encoded_peptides = []

    for peptide in peptides:

        encoded_peptide = []

        for peptide_letter in peptide:
            
            alpha = []

            for alphabet_letter in alphabet:

                alpha.append(encoding_scheme[peptide_letter][alphabet_letter])
                
            encoded_peptide.append(alpha)

        encoded_peptides.append(np.mean(encoded_peptide,axis=0))
        
    return np.array(encoded_peptides)





In [ ]:
alphabet = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V','W', 'Y', 'X']
sum_mat = {}
mat_data = read_sum('../local_snapshot/data/mat_3di.txt')
for i, letter_1 in enumerate(alphabet):
    
    sum_mat[letter_1] = {}

    for j, letter_2 in enumerate(alphabet):
        
        sum_mat[letter_1][letter_2] = mat_data[i, j]


In [ ]:
import pandas as pd

df = pd.read_table('../local_snapshot/tttianhao-CLEAN-0cf2cac/data/new2.csv', header=None)
sequence = df[2]
sequences = encode(sequence, sum_mat, alphabet).tolist()
np.array(sequences).shape

In [ ]:
ID = df[0]
sub_emb = {}
for i,idd in enumerate(ID):
    sub_emb[idd]=sequences[i]

In [ ]:
import h5py
with h5py.File('../local_snapshot/tttianhao-CLEAN-0cf2cac/data/sub3Di/new_sub3Di.h5', "w") as hf:
        for sequence_id, embedding in sub_emb.items():
            # noinspection PyUnboundLocalVariable
            hf.create_dataset(sequence_id, data=embedding)

In [ ]:
from CLEAN.utils import *
dirs = "../local_snapshot/tttianhao-CLEAN-0cf2cac/"
id_ec_dict, ec_id_dict = get_ec_id_dict(dirs+"data/split10_re2.csv")
use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")
dtype = torch.float32

In [ ]:
file = '../local_snapshot/tttianhao-CLEAN-0cf2cac/data/prost5/split10_seq.h5'
import h5py
import torch
def load_prost5(lookup):
    with h5py.File(file, 'r') as hf:
        dataset_name = lookup 
        dataset = hf[dataset_name]  

   
        data = dataset[:]

    torch_data = torch.tensor(data)  
    return torch_data.unsqueeze(0)

def pro_embedding(ec_id_dict, device, dtype):
    '''
    Loading esm embedding in the sequence of EC numbers
    prepare for calculating cluster center by EC
    '''
    pro_emb = []
    # for ec in tqdm(list(ec_id_dict.keys())):
    for ec in list(ec_id_dict.keys()):
        ids_for_query = list(ec_id_dict[ec])
        pro_to_cat = [load_prost5(id) for id in ids_for_query]
        pro_emb = pro_emb + pro_to_cat
    return torch.cat(pro_emb).to(device=device, dtype=dtype)

prost5seq_emb = pro_embedding(ec_id_dict, device, dtype)
#esm_dist = get_dist_map(ec_id_dict, esm_emb, device, dtype)




In [ ]:
prost5seq_emb.shape

In [ ]:
pro_dist = get_dist_map(ec_id_dict, prost5seq_emb, device, dtype)

In [ ]:
pickle.dump(pro_dist, open(dirs+'/data/distance_map/' + 'split10_re2_proSEQ' + '.pkl', 'wb'))
pickle.dump(prost5seq_emb, open(dirs+'/data/distance_map/' + 'split10_re2_proSEQ' + '_esm.pkl', 'wb'))

In [ ]:
!python3 ../local_snapshot/tttianhao-CLEAN-0cf2cac/train-triplet.py --training_data split10_re2 --model_name split10proSEQ_triplet --epoch 2500